<a href="https://colab.research.google.com/github/Shivakathavs/Amirrouh/blob/master/llama_claude_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install llama-cpp-python with CUDA/GPU support
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

# Install Hugging Face Hub to pull the model files directly
!pip install huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 MB 37.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.33-py3-none-linux_x86_64.whl size=154115452 sha256=f10a3ffea610242a771841e62d7498b010c10a52f2d3377d7bc99ddcd8b6f804
  Stored in directory: /root/.cache/pip/wheels/13/fc/a3/e0214eb904e10e712ba0256988708e13188e9eddef8fbc5860
Successfully built llama-cpp-python


In [3]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Download the model from the official Hugging Face repository
model_path = hf_hub_download(
    repo_id="empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF",
    filename="Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf"
)

# Load the model directly into VRAM (n_gpu_layers=-1 moves all layers to GPU)
llm = Llama(
    model_path=model_path,
    n_ctx=4096,         # Adjust context length depending on memory limits
    n_gpu_layers=-1,    # Offload all layers to GPU for maximum execution speed
    n_threads=2         # Number of CPU threads
)


Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.ggu(…):   0%|          | 0.00/5.63G [00:00<?, ?B/s]

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 81152 MiB):
  Device 0: NVIDIA A100-SXM4-80GB, compute capability 8.0, VMM: yes, VRAM: 81152 MiB
llama_model_loader: loaded meta data with 35 key-value pairs and 427 tensors from /root/.cache/huggingface/hub/models--empero-ai--Qwythos-9B-Claude-Mythos-5-1M-GGUF/snapshots/a7ab0f6ee807c165e8374e4906773ca39f5fdff3/Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen35
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwythos 9B Claude Mythos 5 1M
llama_model_loader: - kv   3:                         general.size_label str              = 9.0B
llama_model_loader: - kv   4:                         

In [5]:
# Define your prompt
prompt = "how to merge OB data and trades data from bybit database with correct synhronisation to predict mid price?"

# Format using the standard ChatML template suitable for Qwythos-9B
formatted_prompt = f"<|im_start|>system\nYou are a helpful assistant with advanced reasoning capabilities.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

# Generate output
response = llm(
    formatted_prompt,
    max_tokens=512,
    temperature=0.7,
    top_p=0.9,
    stop=["<|im_end|>"]
)

# Print out the text response
print(response['choices'][0]['text'])


Llama.generate: 18 prefix-match found but partial kv removal not supported, re-evaluating full prompt
CUDA Graph id 33 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reuse

<think>
We need to merge OB data and trades data from the Bybit database with correct synchronization to predict the mid price. Let's break down the problem:

1. **Understand the Data Sources**:
   - OB (Order Book) data: Typically includes bids and asks at different price levels, with volume or quantity.
   - Trades data: Historical trade executions with price, quantity, timestamp, etc.

2. **Understand the Goal**:
   - Predict the mid price (average of the best bid and ask) using both OB and trades data.
   - Ensure correct synchronization (matching timestamps, handling latency, etc.).

3. **Key Challenges**:
   - **Timestamp Alignment**: Bybit provides OB snapshots and trades. Need to match them by time (e.g., using the timestamp from the trade or OB snapshot).
   - **Latency**: OB data might be slightly delayed relative to trades, so need to handle that (e.g., use the latest OB snapshot before the trade timestamp, or interpolate).
   - **Data Integrity**: Ensure that we're merging 